# 01 - Data Cleaning: `fact_production_runs` raw export

**Goal:** turn the messy MES export (`data/raw/fact_production_runs_raw.csv`) into a clean table that matches the `fact_production_runs` schema in `manufacturing_oee_db`.

**Approach (the order matters):**
1. Load everything as text, so pandas can't silently misread messy values
2. Inspect and write down every problem
3. Fix them one by one, and log how many rows each fix touched
4. Validate with business rules
5. Save the cleaned file

Business rules used to validate and repair the data:
- `actual_run_minutes = planned_time_minutes - downtime_minutes` (planned time is always 480)
- `good_units + scrap_units = actual_units_produced`
- `unit_cost` = the part's standard cost (from `dim_parts`)
- `scrap_rework_cost = scrap_units x unit_cost x 0.45`
- `downtime_reason = 'None'` exactly when `downtime_minutes = 0`

## Step 1: Import libraries and load the raw data

In [1]:
import re
import numpy as np
import pandas as pd

RAW = "../data/raw/fact_production_runs_raw.csv"
DIMS = "../data/"
OUT = "../data/cleaned/"

# dtype=str + keep_default_na=False: every value stays exactly as exported (blanks stay "")
raw = pd.read_csv(RAW, dtype=str, keep_default_na=False)

plants   = pd.read_csv(DIMS + "dim_plants.csv")
machines = pd.read_csv(DIMS + "dim_machines.csv")
parts    = pd.read_csv(DIMS + "dim_parts.csv")
shifts   = pd.read_csv(DIMS + "dim_shifts.csv")

print(raw.shape)
raw.head(10)

(115025, 19)


,Run ID,Date,Plant,Machine,Shift,Part,Operator,Planned Time (min),Run Time (min),Downtime (min),Downtime Reason,Planned Units,Actual Units,Good Units,Scrap Units,Unit Cost (INR),Scrap/Rework Cost (INR),Remarks,Export Batch
0,RUN-010001,2024-01-01,PLANT_CHN_01,MCH_CNC_01,Morning,PRT_003,OP-1090,480,467,13,Operator Adj.,110.0,105,104.0,1,3100,"Rs. 1,395",,MES-2024-W01
1,RUN010002,01-01-2024,PLANT_CHN_01,MCH_CNC_01,2,PRT-003,OP-1037,480,406,74,Tool Breakage,109,87,85.0,2,3100,2790,,MES-2024-W01
2,RUN010003,2024-01-01,PLANT_CHN_01,MCH_CNC_01,3,PRT_003,OP-1012,8 hrs,469,11,Operator Adjustment,102,91,90,1,3100,1395,,MES-2024-W01
3,RUN010004,01-01-2024,PLANT_CHN_01,MCH-CNC-02,SHF_01,prt_007,OP-1072,480,468,12,Operator Adjustment,238,200,197,3,1250,1687,,MES-2024-W01
4,RUN010005,01-01-2024,PLANT_CHN_01,MCH_CNC_02,SHF_02,PRT_007,OP-1097,480,470,10,Operator Adjustment,233,215,213,2,1250,1125,,MES-2024-W01
5,RUN010006,2024-01-01,PLANT_CHN_01,MCH_CNC_02,SHF_03,PRT_007,OP-1142,480 min,474,6,Operator Adjustment,215,193,188,5,1250,2812,,MES-2024-W01
6,RUN010007,2024-01-01,plant_chn_01,MCH_CNC_03,SHF_01,PRT_003,OP-1079,480,470,10,Operator Adjustment,86.0,78,78,0,N/A,0,,MES-2024-W01
7,RUN010008,2024-01-01,PLANT_CHN_01,MCH_CNC_03,shf_02,PRT_003,OP-1078,480,477,3,Operator Adjustment,102,95,92.0,3,3100,4185,,MES-2024-W01
8,RUN010009,01-01-2024,PLANT_CHN_01,MCH_CNC_03,SHF_03,PRT_003,OP-1077,480,458,22,Operator adjust,99,90,87,3,3100,4185,,MES-2024-W01
9,RUN010010,2024-01-01,PLANT_CHN_01,MCH_PRS_01,SHF_02,PRT-014,OP-1019,480,449,31,Operator Adj.,271,235,228,7,2300,7245,,MES-2024-W01


## Step 2: Inspect (don't change anything yet)

Look for: junk columns, inconsistent labels, mixed date formats, text in number columns, placeholder values, duplicates.
**Write each problem you find in a markdown cell** - this becomes the "Data Quality Issues" section of your README.

In [2]:
print(raw.columns.tolist())
print("\nBlank cells per column:")
print((raw == "").sum())

['Run ID', 'Date', 'Plant', 'Machine', 'Shift', 'Part', 'Operator', 'Planned Time (min)', 'Run Time (min)', 'Downtime (min)', 'Downtime Reason', 'Planned Units', 'Actual Units', 'Good Units', 'Scrap Units', 'Unit Cost (INR)', 'Scrap/Rework Cost (INR)', 'Remarks', 'Export Batch']

Blank cells per column:
Run ID                          0
Date                            0
Plant                           0
Machine                         0
Shift                           0
Part                            0
Operator                     6825
Planned Time (min)            180
Run Time (min)                337
Downtime (min)                  0
Downtime Reason              3987
Planned Units                   0
Actual Units                    0
Good Units                    329
Scrap Units                     0
Unit Cost (INR)               220
Scrap/Rework Cost (INR)       386
Remarks                    110394
Export Batch                    0
dtype: int64


In [3]:
# Inconsistent labels - same thing written many ways
for col in ["Plant", "Shift", "Downtime Reason"]:
    print(col, "->", raw[col].nunique(), "different spellings")
raw["Plant"].value_counts().head(12)

Plant -> 36 different spellings
Shift -> 18 different spellings
Downtime Reason -> 26 different spellings


Plant
PLANT_MDU_01      13304
PLANT_CHN_01      13199
PLANT_CHN_02      13154
PLANT_BLR_01      13138
PLANT_MAN_01      13132
PLANT_PUN_01      13093
PLANT_CBE_01      12987
 PLANT_CHN_01      1015
 PLANT_PUN_01      1014
 PLANT_CBE_01      1005
 PLANT_CHN_02      1002
 PLANT_BLR_01       993
Name: count, dtype: int64

In [4]:
raw["Downtime Reason"].value_counts()

Downtime Reason
Operator Adjustment      50442
Hydraulic Leak            8071
operator adjustment       6919
Operator adjust           6907
Operator Adj.             6775
OP ADJUSTMENT             6713
Material Shortage         6101
Tool Breakage             5888
                          3987
Scheduled Maintenance     1528
HYDRAULIC LEAK            1098
Hydraulic leakage         1097
Hyd Leak                  1097
hydraulic leak            1062
No Material                863
Tool Break                 838
MATERIAL SHORTAGE          833
Mat. Shortage              820
tool breakage              812
TOOL BREAKAGE              806
material shortage          788
Tool Broken                779
SCHEDULED MAINTENANCE      225
Sched. Maint.              198
scheduled maintenance      194
PM                         184
Name: count, dtype: int64

In [5]:
# Mixed date formats
print(raw["Date"].sample(12, random_state=1).tolist())

# Text hiding in numeric columns
for col in ["Planned Time (min)", "Run Time (min)", "Unit Cost (INR)", "Scrap/Rework Cost (INR)"]:
    nonnum = raw[~raw[col].str.fullmatch(r"\d+")][col]
    print(col, "| non-plain-number values:", len(nonnum), "| examples:", nonnum.unique()[:8])

# Duplicates (same run_id appears more than once)
print("\nDuplicate run ids (before any cleaning):", raw["Run ID"].str.upper().str.replace(r"[^A-Z0-9]", "", regex=True).duplicated().sum())

['2025-02-08', '03-05-2024', '23 Aug 2025', '13-09-2024', '26-09-2024', '30-12-2025', '2025-02-03', '04/02/2025', '10-02-2024', '05/07/2024', '2024-04-26', '2024-05-29']
Planned Time (min) | non-plain-number values: 9547 | examples: <StringArray>
['8 hrs', '480 min', '480.0', 'null', 'N/A', 'NA', '', '-']
Length: 8, dtype: str
Run Time (min) | non-plain-number values: 8893 | examples: <StringArray>
['461 min', '430 min', '465.0', '457 min', '475 min', '', '405.0', '460 min']
Length: 8, dtype: str
Unit Cost (INR) | non-plain-number values: 11576 | examples: <StringArray>
[      'N/A',     '1,250', 'Rs. 1,400',     '820.0', 'Rs. 1,250',    '1850.0',
     '1,850', 'Rs. 3,100']
Length: 8, dtype: str
Scrap/Rework Cost (INR) | non-plain-number values: 12072 | examples: <StringArray>
[ 'Rs. 1,395',  'Rs. 9,315',     '2205.0',       'null',      '4,860',
 'Rs. 11,340',      '5,062',     '6615.0']
Length: 8, dtype: str

Duplicate run ids (before any cleaning): 1363


## Step 3: Rename columns and drop junk columns
Operator, remarks and export batch are not part of the database schema.

In [6]:
rename = {
    "Run ID": "run_id", "Date": "date_raw", "Plant": "plant_id", "Machine": "machine_id",
    "Shift": "shift_id", "Part": "part_id", "Operator": "operator_id",
    "Planned Time (min)": "planned_time_minutes", "Run Time (min)": "actual_run_minutes",
    "Downtime (min)": "downtime_minutes", "Downtime Reason": "downtime_reason",
    "Planned Units": "planned_units", "Actual Units": "actual_units_produced",
    "Good Units": "good_units", "Scrap Units": "scrap_units",
    "Unit Cost (INR)": "unit_cost", "Scrap/Rework Cost (INR)": "scrap_rework_cost",
    "Remarks": "remarks", "Export Batch": "export_batch",
}
df = raw.rename(columns=rename).drop(columns=["operator_id", "remarks", "export_batch"])

for c in df.columns:                      # trim leading/trailing spaces everywhere
    df[c] = df[c].str.strip()

log = {"01 raw rows loaded": len(df)}
df.head()

,run_id,date_raw,plant_id,machine_id,shift_id,part_id,planned_time_minutes,actual_run_minutes,downtime_minutes,downtime_reason,planned_units,actual_units_produced,good_units,scrap_units,unit_cost,scrap_rework_cost
0,RUN-010001,2024-01-01,PLANT_CHN_01,MCH_CNC_01,Morning,PRT_003,480,467,13,Operator Adj.,110.0,105,104.0,1,3100,"Rs. 1,395"
1,RUN010002,01-01-2024,PLANT_CHN_01,MCH_CNC_01,2,PRT-003,480,406,74,Tool Breakage,109,87,85.0,2,3100,2790
2,RUN010003,2024-01-01,PLANT_CHN_01,MCH_CNC_01,3,PRT_003,8 hrs,469,11,Operator Adjustment,102,91,90,1,3100,1395
3,RUN010004,01-01-2024,PLANT_CHN_01,MCH-CNC-02,SHF_01,prt_007,480,468,12,Operator Adjustment,238,200,197,3,1250,1687
4,RUN010005,01-01-2024,PLANT_CHN_01,MCH_CNC_02,SHF_02,PRT_007,480,470,10,Operator Adjustment,233,215,213,2,1250,1125


## Step 4: Standardize `run_id` and remove duplicates
Standardize **first** (`run-010001`, ` RUN010001` and `RUN010001` are the same run), then de-duplicate on `run_id`.

In [7]:
df["run_id"] = df["run_id"].str.upper().str.replace(r"[^A-Z0-9]", "", regex=True)
assert df["run_id"].str.fullmatch(r"RUN\d{6}").all()

n = len(df)
df = df.drop_duplicates("run_id", keep="first").copy()
log["02 duplicate rows removed"] = n - len(df)
print(log)

{'01 raw rows loaded': 115025, '02 duplicate rows removed': 1363}


## Step 5: Parse the dates
Five formats are mixed. Parsing each format **explicitly** is safer than `dayfirst=True`, which can swap day and month on `YYYY-MM-DD` dates.
Rows with impossible dates (e.g. 31-02-2024) or dates outside 2024-2025 can't be trusted, so we drop them.

In [8]:
DATE_FMTS = ["%d-%m-%Y", "%Y-%m-%d", "%d/%m/%Y", "%d %b %Y", "%d-%b-%y"]

parsed = pd.Series(pd.NaT, index=df.index, dtype="datetime64[ns]")
for f in DATE_FMTS:
    parsed = parsed.fillna(pd.to_datetime(df["date_raw"], format=f, errors="coerce"))

unparsed = parsed.isna()
out_of_range = ~unparsed & ~parsed.between("2024-01-01", "2025-12-31")
print("Unparseable dates:", unparsed.sum(), "| outside 2024-2025:", out_of_range.sum())
print(df.loc[unparsed | out_of_range, "date_raw"].value_counts().head(8))

df["date_id"] = parsed.dt.strftime("%Y-%m-%d")
df = df[~(unparsed | out_of_range)].drop(columns="date_raw").copy()
log["03 rows dropped: bad / out-of-range date"] = int((unparsed | out_of_range).sum())

Unparseable dates: 30 | outside 2024-2025: 25
date_raw
31-02-2024    30
12-01-2042     1
13-02-2023     1
27-02-2026     1
20-03-2023     1
13-04-2026     1
17-04-2042     1
22-04-2026     1
Name: count, dtype: int64


## Step 6: Standardize the keys (plant, machine, part, shift) and drop orphans
Every key must exist in its dimension table, otherwise the star schema breaks (and Power BI relationships will show blanks).

In [9]:
df["plant_id"]   = df["plant_id"].str.upper().str.replace(r"[\s\-]+", "_", regex=True)
df["machine_id"] = df["machine_id"].str.upper().str.replace(r"[\s\-]+", "_", regex=True)
df["part_id"]    = df["part_id"].str.upper().str.replace(r"^PRT[\s_\-]*(\d{3})$", r"PRT_\1", regex=True)

def std_shift(x):
    x = x.lower()
    for name, no in {"morning": 1, "evening": 2, "night": 3}.items():
        if name in x:
            return f"SHF_0{no}"
    m = re.search(r"(\d+)", x)
    return f"SHF_0{int(m.group(1))}" if m else None

df["shift_id"] = df["shift_id"].map(std_shift)

valid = (df.plant_id.isin(plants.plant_id) & df.machine_id.isin(machines.machine_id)
         & df.part_id.isin(parts.part_id) & df.shift_id.isin(shifts.shift_id))
print("Rows with keys not in the dimension tables:", (~valid).sum())
print(df.loc[~valid, ["plant_id", "machine_id"]].value_counts().head())

df = df[valid].copy()
log["04 rows dropped: unknown plant / machine key"] = int((~valid).sum())

Rows with keys not in the dimension tables: 173
plant_id      machine_id
PLANT_MAN_01  MCH_XXX_99    24
PLANT_XYZ_09  MCH_HTR_01    17
              MCH_CNC_02    17
PLANT_CHN_01  MCH_XXX_99    13
PLANT_XYZ_09  MCH_CNC_01    12
Name: count, dtype: int64


## Step 7: Standardize downtime reasons
Map every spelling to one of the 6 official categories. Blank / `-` / `NIL` / `N/A` all mean *no downtime* -> `None`.

In [10]:
reason_map = {}
for canon, variants in {
    "Operator Adjustment":   ["operator adjustment", "operator adj.", "op adjustment", "operator adjust"],
    "Tool Breakage":         ["tool breakage", "tool break", "tool broken"],
    "Hydraulic Leak":        ["hydraulic leak", "hyd leak", "hydraulic leakage"],
    "Material Shortage":     ["material shortage", "mat. shortage", "no material"],
    "Scheduled Maintenance": ["scheduled maintenance", "sched. maint.", "pm"],
    "None":                  ["none", "nil", "-", "", "n/a", "na"],
}.items():
    for v in variants:
        reason_map[v] = canon

df["downtime_reason"] = df["downtime_reason"].str.lower().str.strip().map(reason_map)
assert df["downtime_reason"].notna().all(), "unmapped reason found - add it to reason_map"
df["downtime_reason"].value_counts()

downtime_reason
Operator Adjustment      76679
Hydraulic Leak           12267
Material Shortage         9272
Tool Breakage             8993
None                      3930
Scheduled Maintenance     2293
Name: count, dtype: int64

## Step 8: Convert the numeric columns
Strip units and symbols (`min`, `Rs.`, commas, `8 hrs`), turn placeholders (`N/A`, `null`, `-`) into proper missing values (`NaN`).

In [11]:
NUM_COLS = ["planned_time_minutes", "actual_run_minutes", "downtime_minutes", "planned_units",
            "actual_units_produced", "good_units", "scrap_units", "unit_cost", "scrap_rework_cost"]

def to_num(s):
    s = s.str.lower().str.strip().replace({"n/a": "", "na": "", "null": "", "-": "", "nan": ""})
    hours = s.str.contains("hr", na=False)                       # "8 hrs" -> 480 minutes
    s = s.str.replace(r"rs\.?|inr|\u20b9|mins?|hrs?|,", "", regex=True).str.strip()
    out = pd.to_numeric(s, errors="coerce")
    return out.where(~hours, out * 60)

for c in NUM_COLS:
    df[c] = to_num(df[c])

print("Missing values after conversion:")
print(df[NUM_COLS].isna().sum())

Missing values after conversion:
planned_time_minutes     430
actual_run_minutes       871
downtime_minutes           0
planned_units              0
actual_units_produced      0
good_units               875
scrap_units                0
unit_cost                572
scrap_rework_cost        928
dtype: int64


## Step 9: Repair values using business rules
Each fix uses a rule instead of guessing. Count how many rows each fix touches and keep it in the log.

In [12]:
# 9.1 Planned time is always 480 (a shift)
assert df["planned_time_minutes"].dropna().eq(480).all()
log["05 fixed: planned time blank -> 480"] = int(df["planned_time_minutes"].isna().sum())
df["planned_time_minutes"] = df["planned_time_minutes"].fillna(480)

# 9.2 Negative scrap = sign typo
log["06 fixed: negative scrap units"] = int((df["scrap_units"] < 0).sum())
df["scrap_units"] = df["scrap_units"].abs()

# 9.3 Run time and downtime must add up to planned time
#     If run time is missing -> derive it from downtime. Otherwise trust run time and re-derive downtime.
run_missing = df["actual_run_minutes"].isna()
inconsistent = ~run_missing & (df["actual_run_minutes"] + df["downtime_minutes"] != df["planned_time_minutes"])
log["07 fixed: run time blank"] = int(run_missing.sum())
log["08 fixed: downtime inconsistent with run time"] = int(inconsistent.sum())
df["actual_run_minutes"] = df["actual_run_minutes"].fillna(df["planned_time_minutes"] - df["downtime_minutes"])
df["downtime_minutes"] = df["planned_time_minutes"] - df["actual_run_minutes"]

# 9.4 Actual units can't be far above plan (a 10x typo) -> rebuild from good + scrap
bad_actual = df["actual_units_produced"] > df["planned_units"] * 1.25
log["09 fixed: actual units 10x typo"] = int(bad_actual.sum())
df.loc[bad_actual, "actual_units_produced"] = df["good_units"] + df["scrap_units"]

# 9.5 Good units = actual - scrap (fixes blanks and 10x typos in good units)
bad_good = df["good_units"].isna() | (df["good_units"] != df["actual_units_produced"] - df["scrap_units"])
log["10 fixed: good units blank / wrong"] = int(bad_good.sum())
df["good_units"] = df["actual_units_produced"] - df["scrap_units"]

# 9.6 Unit cost comes from the part master
std_cost = parts.set_index("part_id")["standard_unit_cost"]
log["11 fixed: unit cost blank"] = int(df["unit_cost"].isna().sum())
log["12 unit cost that disagreed with part master"] = int((df["unit_cost"].notna() & (df["unit_cost"] != df["part_id"].map(std_cost))).sum())
df["unit_cost"] = df["part_id"].map(std_cost)

# 9.7 Scrap/rework cost = scrap x unit cost x 0.45
calc_cost = (df["scrap_units"] * df["unit_cost"] * 0.45).astype(int)
log["13 fixed: scrap cost blank"] = int(df["scrap_rework_cost"].isna().sum())
log["14 scrap cost that disagreed with the formula"] = int((df["scrap_rework_cost"].notna() & (df["scrap_rework_cost"] != calc_cost)).sum())
df["scrap_rework_cost"] = calc_cost

df[NUM_COLS] = df[NUM_COLS].astype(int)
pd.Series(log)

01 raw rows loaded                               115025
02 duplicate rows removed                          1363
03 rows dropped: bad / out-of-range date             55
04 rows dropped: unknown plant / machine key        173
05 fixed: planned time blank -> 480                 430
06 fixed: negative scrap units                      220
07 fixed: run time blank                            871
08 fixed: downtime inconsistent with run time       214
09 fixed: actual units 10x typo                      44
10 fixed: good units blank / wrong                  910
11 fixed: unit cost blank                           572
12 unit cost that disagreed with part master          0
13 fixed: scrap cost blank                          928
14 scrap cost that disagreed with the formula         0
dtype: int64

## Step 10: Validate - the cleaned data must satisfy every business rule

In [13]:
assert df["run_id"].is_unique
assert df[NUM_COLS].ge(0).all().all()
assert (df["actual_run_minutes"] + df["downtime_minutes"] == df["planned_time_minutes"]).all()
assert (df["good_units"] + df["scrap_units"] == df["actual_units_produced"]).all()
assert ((df["downtime_reason"] == "None") == (df["downtime_minutes"] == 0)).all()
assert (df["actual_units_produced"] <= df["planned_units"] * 1.25).all()
assert (df["scrap_rework_cost"] == (df["scrap_units"] * df["unit_cost"] * 0.45).astype(int)).all()
assert not df.duplicated(["date_id", "plant_id", "machine_id", "shift_id"]).any()
print("All validation checks passed.", len(df), "rows")

# quick sanity check on the business meaning
avail = df.actual_run_minutes.sum() / df.planned_time_minutes.sum()
perf  = df.actual_units_produced.sum() / (df.planned_units * df.actual_run_minutes / df.planned_time_minutes).sum()
qual  = df.good_units.sum() / df.actual_units_produced.sum()
print(f"Availability {avail:.1%} | Performance {perf:.1%} | Quality {qual:.1%} | OEE {avail*perf*qual:.1%}")

All validation checks passed. 113434 rows
Availability 93.4% | Performance 91.5% | Quality 96.9% | OEE 82.8%


## Step 11: Reconcile against the source database (learning check)
In a real job there is no answer key. Here the original clean table exists, so we can **prove** the cleaning worked: every kept row should match it exactly.

In [14]:
cols = ["run_id", "date_id", "plant_id", "machine_id", "shift_id", "part_id", "planned_time_minutes",
        "actual_run_minutes", "downtime_minutes", "downtime_reason", "planned_units",
        "actual_units_produced", "good_units", "scrap_units", "unit_cost", "scrap_rework_cost"]
clean = df[cols].sort_values("run_id").reset_index(drop=True)

# keep_default_na=False: the text "None" in downtime_reason is a real category, but pandas would read it as missing (NaN)
truth = pd.read_csv(DIMS + "fact_production_runs.csv", keep_default_na=False)
m = clean.merge(truth, on="run_id", how="left", suffixes=("", "_true"))

print("Rows not found in source:", m["date_id_true"].isna().sum())
mismatch = {c: int((m[c].astype(str) != m[c + "_true"].astype(str)).sum()) for c in cols if c != "run_id"}
print("Mismatching values per column:", mismatch)
print(f"Source rows: {len(truth):,} | cleaned rows: {len(clean):,} | lost: {len(truth) - len(clean):,}")

Rows not found in source: 0
Mismatching values per column: {'date_id': 0, 'plant_id': 0, 'machine_id': 0, 'shift_id': 0, 'part_id': 0, 'planned_time_minutes': 0, 'actual_run_minutes': 0, 'downtime_minutes': 0, 'downtime_reason': 0, 'planned_units': 0, 'actual_units_produced': 0, 'good_units': 0, 'scrap_units': 0, 'unit_cost': 0, 'scrap_rework_cost': 0}
Source rows: 113,662 | cleaned rows: 113,434 | lost: 228


## Step 12: Save the cleaned data

**Gotcha for later notebooks:** `downtime_reason` contains the real text value `None`. When you reload this file in pandas use `pd.read_csv(..., keep_default_na=False)`, otherwise `None` silently becomes `NaN` and a whole downtime category disappears from your analysis.

In [15]:
import os
os.makedirs(OUT, exist_ok=True)
clean.to_csv(OUT + "fact_production_runs_clean.csv", index=False)

summary = pd.Series(log, name="rows affected").to_frame()
summary.to_csv(OUT + "cleaning_log.csv")
summary

,rows affected
01 raw rows loaded,115025
02 duplicate rows removed,1363
03 rows dropped: bad / out-of-range date,55
04 rows dropped: unknown plant / machine key,173
05 fixed: planned time blank -> 480,430
06 fixed: negative scrap units,220
07 fixed: run time blank,871
08 fixed: downtime inconsistent with run time,214
09 fixed: actual units 10x typo,44
10 fixed: good units blank / wrong,910


## Next: load into MySQL

Load `data/cleaned/fact_production_runs_clean.csv` into the `fact_production_runs` table (MySQL Workbench > Table Data Import Wizard, or `pandas.to_sql` with SQLAlchemy), then continue with the SQL analysis.

**Add your own "Cleaning Summary" to the README:** issue found -> rows affected (from the log above) -> how you fixed it -> why that decision was safe.